In [75]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-07 19:36:32--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8002::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 416 Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [76]:
with open('input.txt', 'r' , encoding='utf-8') as f:
    text = f.read()

In [77]:
print("length of dataset in characters :" , len(text))

length of dataset in characters : 1115394


In [78]:
#here are all the unique characters that occure in this text

chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [79]:
# tokenising the input text ---> character level tokenizer

stoi = {ch:i for i , ch in enumerate(chars)}
itos = { i:ch for i , ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode: take a string , output a list of  integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder : take a list of integers, output a string

print(encode("hi there"))
print(decode(encode("hi there")))


[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [80]:
# encode the entire dataset
import torch
data = torch.tensor(encode(text) , dtype=torch.long)
data.shape , data.dtype

(torch.Size([1115394]), torch.int64)

In [81]:
data[:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [82]:
# lets new split up the data into train and test split

n = int(0.9*len(data))
train_data = data[:n] # 90% will be train 
val_data = data[:n] # 10% will be val 

In [83]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [84]:
x = train_data[:block_size]  # x is the input to the transformers 
y = train_data[:block_size+1] # y is the next block size character 
print(x , y)

for t in range  (block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context} the target : {target}")

tensor([18, 47, 56, 57, 58,  1, 15, 47]) tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])
When input is tensor([18]) the target : 18
When input is tensor([18, 47]) the target : 47
When input is tensor([18, 47, 56]) the target : 56
When input is tensor([18, 47, 56, 57]) the target : 57
When input is tensor([18, 47, 56, 57, 58]) the target : 58
When input is tensor([18, 47, 56, 57, 58,  1]) the target : 1
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the target : 15
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target : 47


In [85]:
torch.manual_seed(1337)
batch_size = 4 # how many indepedent sequences will be processed in parallel
block_size = 8 # what is the maximum context length for predictions


def get_batch(split):
    # generate a small batch of data of inputs x and target y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size , (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x , y

xb , yb = get_batch('train')

print("inputs:")
print(xb.shape)
print(xb)
print('traget:')
print(yb.shape)
print(yb)

print('__________')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
traget:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
__________
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [4

#  Bigram Language model 

In [39]:
import torch
import torch.nn as nn
from torch.nn import  functional as F
torch.manual_seed(1337)

In [96]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size , vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


    def generate(self, idx , max_new_tokens):
        for _ in range(max_new_tokens):
            # get the predictions 
            logits , loss = self(idx)
            # focus only on the last time step
            logits  = logits[: ,-1,:]
            # apply softmax to get probabilites
            probs = F.softmax(logits , dim=-1) # (b,c)
            # sample from the distributions 
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence 
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



In [97]:
model = BigramLanguageModel(vocab_size)
logits , loss = model(xb , yb)
print(out.shape)
print(loss)

torch.Size([4, 9, 65])
tensor(4.9095, grad_fn=<NllLossBackward0>)


In [98]:

print(decode(model.generate(idx = torch.zeros((1,1) ,dtype=torch.long), max_new_tokens=100)[0].tolist()))


NZqlz':Pk.RPmfk3z3yDvH-&&XbIzNeblzfavPPAv'wvHx!P&
Tj
dpnFH,PgcCqiqOPVGWNLwmBwOzg-Mjj
c!Cher.lnFf!gAp


# train model

In [100]:
# pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [103]:
batch_size=32
for steps in range(10000):

    # sample the batch data

    xb , yb = get_batch('train')


    # eval the loss

    logits , loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5453574657440186


`loss.item()` is a noisy cause every batch is more or less lucky , so we use `estimate_loss()` in our py version of the code . It average up the loss over the multiple batches 

In [106]:
# sample the model
print(decode(model.generate(idx = torch.zeros((1,1) ,dtype=torch.long), max_new_tokens=500)[0].tolist()))


I to hathely beriser?
OLESS:
FI t hy sondit mbyptodsuimy.

Wildssso, yo ERInoware henck. lowathaso-
I:
I ie
RFRI hteeliouret ar, hatwel thisere f alou,
WASo s; patillfot by,
Thtoof IXET:
WI thy we, wes ad,'sler wo, ale t lll bun p,
co w t me f as, llor ch y PEN le:
Wan Myot pr tyfr ter'than fat; avireto lshondo.
INRYond Wh l d,
Who beserpurids wayout lowat'de.

Aniaty ty f e MIf, pralito nd awe sindrnd what nu chain?
Seas thys
O: w ppeito;

d k t he milode G grce iove yours rime ithivinse g heng


>Our model output is so random cause the token is not talking to each other , or we can say token are not give attentions to each other (ignoring the srrounding token)

## The mathematical trick in self attention


- Causal self-attention or masked self attention 

In this setup the mechanism is designed so that a token a a specific postion can only interact with or pay attention to tokens that came before aka past tokens , This is crucial or autoregressive models like GPT , to generate text sequentially without looking ahead at the future token or information .


- What is the easiest way for token to communicate ?

If we're a fifth token and I'd like to communicate With my past , the simplest we can do is just to average of preceding elements (past tokens) . we essentially assigning the exact same weight or importance to each previous token. This mean the past token contributes an equal amount of influence to the current position's context.
> By doing the average is an extremely weak form of interaction , this communication is extermly lossy 

In [33]:
# a toy example 
import torch 
import torch.nn.functional as F
torch.manual_seed(1337)
B, T, C = 4, 8 , 2 # batch , time , channels
x = torch.randn(B, T, C)
x.shape 

torch.Size([4, 8, 2])

In [25]:
# we want x[b,t] = mean_{i<=t} x[b,i]

xbow = torch.zeros((B,T,C)) #bow - bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0) 

In [31]:
# version 2

wei = torch.tril(torch.ones(T, T))
wei = wei/wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B,T,C) ---- >(B,T,C)

torch.allclose(xbow, xbow2 , atol=1e-7)

#  atol=1e-7 adding this in `torch.allclose()` wont affect anything its just to remove the floating point error

True

In [29]:
xbow[0] == xbow2[0]

tensor([[True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True]])

In [35]:
# version 3 : use softmax

tril  = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0 , float('-inf'))
wei = F.softmax(wei , dim=1)
xbow3 = wei @ x
torch.allclose(xbow , xbow3 , atol=1e-7)

True

In [36]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [45]:
# version 4 : self-attention

torch.manual_seed(1337)
B,T,C = 4,8,32 
x = torch.randn(B,T,C)

# lets see a single head perform self attention 
head_size = 16
key = nn.Linear(C, head_size , bias=False)
query = nn.Linear(C, head_size ,bias = False)
value = nn.Linear(C, head_size , bias=False)

k  = key(x) # (B,T,16)
q = query(x) # (B,T,16)

wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B , 16, T) ---> (B,T,T) 

tril =  torch.tril(torch.ones(T,T))
# wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0 , float('-inf'))
wei = F.softmax(wei , dim=-1)
# out = wei @ x
v = value(x)
out = wei @ v

out.shape


torch.Size([4, 8, 16])

In [46]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [47]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [13]:
"""
    here the first token is same in x[0] and xbow[0] position the 2nd token in xbow[0] is the average of 1st and 2nd
    token in x[0] goes so on till the last token , where the last token in xbow[0] is sum of all the previous token in x[0]


    in efficent - best way is matrix multiplication
"""

x[0] , xbow[0]

(tensor([[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [18]:
# example with matrix multiplication 

torch.manual_seed(42)

a = torch.tril(torch.ones(3,3))
a=a/torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()

c = a @ b


print(f'a=',a)
print('------')
print(f'b=', b)
print('------')
print(f'c=' , c)

a= tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
------
b= tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
------
c= tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

